# Pathology detector v3 — 9,848 images

Same architecture and hyperparameters as v2 (`yolov8s`, 1024 px, identical
augmentation) so the only variable is the training data:

| | v2 | v3 |
|---|---|---|
| DENTEX 4-class | 598 imgs / 3,031 boxes | 598 imgs / 3,031 boxes |
| OralXrays-9 (3-class, remapped) | — | 9,250 imgs / 36,186 boxes |
| classes | 4 | 4 (OralXrays `decay` -> `Caries`) |

Deep Caries exists only in DENTEX, so those images are oversampled 2x.
Validation stays the same 107-image DENTEX split as v2.

HuggingFace allows 3,000 resolver requests per 5 minutes per IP.
`snapshot_download` issues HEAD+GET per file for 18,500 files and bursts past
that limit, then backs off 193 s per retry, which never converges. The download
cell below does one GET per file at 9 req/s instead, which finishes in ~40 min.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# ultralytics + weights come from the private wheels dataset (no dependency
# resolution, known-good torch). This kernel needs internet for the HF mirror.
WHEELS = next(Path("/kaggle/input").rglob("yolov8s.pt")).parent
os.environ["YOLO_OFFLINE"] = "True"

try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                    "--find-links", str(WHEELS), "ultralytics", "ultralytics-thop"],
                   check=True)
    import ultralytics

import torch
print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU available. Kaggle silently runs CPU-only until the account is "
        "phone-verified: https://www.kaggle.com/settings -> Phone Verification."
    )
p = torch.cuda.get_device_properties(0)
print("gpu:", p.name, f"{p.total_memory / 1e9:.1f} GB")

In [ ]:
import json
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
SEED, VAL_FRAC = 42, 0.15

DISEASE_NAMES = ["Impacted", "Caries", "Periapical Lesion", "Deep Caries"]


def load_coco(path):
    with open(path) as fh:
        return json.load(fh)


def image_map(coco):
    return {i["id"]: i for i in coco["images"]}


def yolo_labels(coco, class_fn):
    imgs = image_map(coco)
    out = defaultdict(list)
    for ann in coco["annotations"]:
        img = imgs[ann["image_id"]]
        W, H = img["width"], img["height"]
        x, y, w, h = ann["bbox"]
        x1, y1 = max(0.0, x), max(0.0, y)
        x2, y2 = min(W, x + w), min(H, y + h)
        w, h = x2 - x1, y2 - y1
        if w <= 1 or h <= 1:
            continue
        out[img["file_name"]].append(
            f"{class_fn(ann)} {(x1 + w / 2) / W:.6f} {(y1 + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}")
    return dict(out)


def stratified_split(names, signature_fn, val_frac=VAL_FRAC, seed=SEED):
    groups = defaultdict(list)
    for n in names:
        groups[signature_fn(n)].append(n)
    rng = random.Random(seed)
    train, val = [], []
    for key in sorted(groups):
        items = sorted(groups[key])
        rng.shuffle(items)
        n_val = max(1, int(round(len(items) * val_frac))) if len(items) > 1 else 0
        val.extend(items[:n_val])
        train.extend(items[n_val:])
    return sorted(train), sorted(val)


def build(root, name, src_dir, labels, train_names, val_names, class_names):
    root = root / name
    for split, names in (("train", train_names), ("val", val_names)):
        (root / "images" / split).mkdir(parents=True, exist_ok=True)
        (root / "labels" / split).mkdir(parents=True, exist_ok=True)
        for n in names:
            dst = root / "images" / split / n
            if not dst.exists():
                os.symlink(src_dir / n, dst)
            (root / "labels" / split / (Path(n).stem + ".txt")).write_text(
                "\n".join(labels.get(n, [])) + ("\n" if labels.get(n) else ""))
    (root / "data.yaml").write_text(
        f"path: {root}\ntrain: images/train\nval: images/val\n\n"
        f"nc: {len(class_names)}\nnames: {class_names}\n")
    return root

In [ ]:
# ---- DENTEX 4-class split (identical to v2: 598 / 107) ----
qed_json = next(INPUT.rglob("train_quadrant_enumeration_disease.json"))
qed = load_coco(qed_json)
qed_imgs = qed_json.parent / "xrays"
print("annotations:", qed_json)
print("images:", len(qed["images"]), "| boxes:", len(qed["annotations"]))

labels = yolo_labels(qed, lambda a: a["category_id_3"])

by_image = defaultdict(set)
for ann in qed["annotations"]:
    by_image[ann["image_id"]].add(ann["category_id_3"])
id_to_name = {i["id"]: i["file_name"] for i in qed["images"]}


def signature(name):
    ids = next((v for k, v in by_image.items() if id_to_name[k] == name), set())
    return tuple(sorted(ids))


train_names, val_names = stratified_split([i["file_name"] for i in qed["images"]], signature)
root = build(WORK / "datasets", "disease", qed_imgs, labels, train_names, val_names, DISEASE_NAMES)

n_train_boxes = sum(len(labels.get(n, [])) for n in train_names)
n_val_boxes = sum(len(labels.get(n, [])) for n in val_names)
print(f"train {len(train_names)} imgs / {n_train_boxes} boxes")
print(f"val   {len(val_names)} imgs / {n_val_boxes} boxes")
assert len(train_names) == 598 and len(val_names) == 107, "split drifted from the locked build"
assert n_train_boxes == 3031 and n_val_boxes == 498, f"box counts drifted: {n_train_boxes}/{n_val_boxes}"

In [ ]:
# ---- download OralXrays-9 (9,250 jpgs + labels) at a rate HF tolerates ----
import threading
import time
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed

HF = WORK / "hf_oral"
BASE = "https://huggingface.co/datasets/liodon-ai/dental-panoramic-xray-yolo/resolve/main/"
API = "https://huggingface.co/api/datasets/liodon-ai/dental-panoramic-xray-yolo/tree/main/"


def list_dir(path):
    out, url = [], API + path
    while url:
        req = urllib.request.Request(url, headers={"User-Agent": "kaggle"})
        with urllib.request.urlopen(req, timeout=60) as r:
            out += [f["path"] for f in json.loads(r.read()) if f["type"] == "file"]
            link = r.headers.get("Link") or ""
        url = link.split("<")[1].split(">")[0] if 'rel="next"' in link else None
    return out


paths = [p for p in list_dir("images/train") + list_dir("labels/train")
         if p.endswith((".jpg", ".txt"))]
print("files to fetch:", len(paths))

_lock = threading.Lock()
_next = [0.0]
RATE = 9.0  # resolver cap is 10/s sustained


def pace():
    with _lock:
        now = time.monotonic()
        _next[0] = max(_next[0], now)
        wait = _next[0] - now
        _next[0] += 1.0 / RATE
    if wait > 0:
        time.sleep(wait)


def fetch(path):
    dst = HF / path
    if dst.exists() and dst.stat().st_size > 0:
        return "skip"
    dst.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(8):
        pace()
        try:
            req = urllib.request.Request(BASE + path, headers={"User-Agent": "kaggle"})
            with urllib.request.urlopen(req, timeout=180) as r:
                data = r.read()
            tmp = dst.with_suffix(dst.suffix + ".part")
            tmp.write_bytes(data)
            tmp.replace(dst)
            return "ok"
        except urllib.error.HTTPError as e:
            if e.code == 429:
                time.sleep(float(e.headers.get("Retry-After") or 30))
                continue
            if e.code in (500, 502, 503, 504):
                time.sleep(5 * (attempt + 1))
                continue
            return f"err{e.code}"
        except Exception:
            time.sleep(3 * (attempt + 1))
    return "failed"


t0 = time.time()
done, fails = 0, []
with ThreadPoolExecutor(max_workers=12) as ex:
    futs = {ex.submit(fetch, p): p for p in paths}
    for fut in as_completed(futs):
        status = fut.result()
        done += 1
        if status not in ("ok", "skip"):
            fails.append((futs[fut], status))
        if done % 1000 == 0:
            print(f"{done}/{len(paths)}  {done / (time.time() - t0):.1f}/s  "
                  f"failures {len(fails)}", flush=True)

print(f"downloaded in {(time.time() - t0) / 60:.1f} min, failures {len(fails)}")
assert not fails, fails[:10]

oral_imgs = sorted((HF / "images" / "train").glob("*.jpg"))
oral_labs = HF / "labels" / "train"
print("oral images:", len(oral_imgs), "| label files:", len(list(oral_labs.glob("*.txt"))))
assert len(oral_imgs) == 9250, f"expected 9,250 oral images, found {len(oral_imgs)}"

In [ ]:
# ---- remap OralXrays 3 classes onto the 4-class index, in place ----
# HF: 0=caries, 1=periapical, 2=impacted
# ours: 0=Impacted, 1=Caries, 2=Periapical Lesion, 3=Deep Caries
HF_MAP = {0: 1, 1: 2, 2: 0}
marker = HF / ".remapped_v3"

if not marker.exists():
    n_files = 0
    for lab in oral_labs.glob("*.txt"):
        out = []
        for line in lab.read_text().splitlines():
            if not line.strip():
                continue
            parts = line.split()
            parts[0] = str(HF_MAP[int(parts[0])])
            out.append(" ".join(parts))
        lab.write_text("\n".join(out) + ("\n" if out else ""))
        n_files += 1
    marker.write_text("ok")
    print("remapped", n_files, "label files")
else:
    print("already remapped, skipping")

oral_class_counts = Counter()
for lab in oral_labs.glob("*.txt"):
    for line in lab.read_text().splitlines():
        if line.strip():
            oral_class_counts[DISEASE_NAMES[int(line.split()[0])]] += 1
print("oral class distribution:")
for k, v in oral_class_counts.most_common():
    print(f"  {k:22s} {v:6d}")
assert oral_class_counts["Impacted"] > 20000 and oral_class_counts["Caries"] > 10000

In [ ]:
# ---- combined train list: DENTEX (4-class) + OralXrays (remapped) ----
dentex_train = [str(root / "images" / "train" / n) for n in train_names]
dentex_val = [str(root / "images" / "val" / n) for n in val_names]
oral_train = [str(p) for p in oral_imgs if (oral_labs / (p.stem + ".txt")).exists()]

# Deep Caries only exists in DENTEX (578 boxes), so oversample those images 2x.
deep = {n for n in train_names if any(l.startswith("3 ") for l in labels.get(n, []))}
dentex_train_os = [p for p in dentex_train for _ in (range(2) if Path(p).name in deep else range(1))]

train_list = dentex_train_os + oral_train
val_list = dentex_val

(WORK / "datasets" / "train_v3.txt").write_text("\n".join(train_list) + "\n")
(WORK / "datasets" / "val_v3.txt").write_text("\n".join(val_list) + "\n")
(WORK / "datasets" / "data_v3.yaml").write_text(
    f"path: {WORK / 'datasets'}\ntrain: train_v3.txt\nval: val_v3.txt\n\n"
    f"nc: 4\nnames: {DISEASE_NAMES}\n")

print(f"train entries {len(train_list)}  ({len(dentex_train)} dentex + "
      f"{len(dentex_train_os) - len(dentex_train)} deep-caries repeats + {len(oral_train)} oral)")
print(f"val entries   {len(val_list)}")

In [ ]:
# ---- sanity checks before burning GPU time ----
import numpy as np

missing_img = [p for p in train_list + val_list if not Path(p).exists()]
missing_lab = [p for p in train_list + val_list
               if not Path(str(p).replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt").exists()]
print("missing images:", len(missing_img), "| missing labels:", len(missing_lab))
assert not missing_img and not missing_lab

train_stems = {Path(p).stem for p in train_list}
val_stems = {Path(p).stem for p in val_list}
print("train/val stem overlap:", len(train_stems & val_stems))
assert not (train_stems & val_stems), "train and val overlap"

boxes = Counter()
per_img = []
for p in train_list:
    lab = Path(str(p).replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt")
    lines = [l for l in lab.read_text().splitlines() if l.strip()]
    per_img.append(len(lines))
    for l in lines:
        boxes[DISEASE_NAMES[int(l.split()[0])]] += 1
print("\ntrain class distribution:")
for k, v in boxes.most_common():
    print(f"  {k:22s} {v:6d}")
print(f"\nboxes/img: mean {np.mean(per_img):.2f}  max {max(per_img)}  "
      f"images with 0 boxes {sum(1 for x in per_img if x == 0)}")
print("total boxes:", sum(boxes.values()))

In [ ]:
from ultralytics import YOLO

IMGSZ = 1024
EPOCHS = 40
BATCH = 8
MODEL = str(WHEELS / "yolov8s.pt")

model = YOLO(MODEL)
results = model.train(
    data=str(WORK / "datasets" / "data_v3.yaml"),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    patience=10,
    seed=SEED,
    deterministic=True,
    optimizer="auto",
    cos_lr=True,
    close_mosaic=10,
    amp=True,
    workers=2,
    project=str(WORK / "runs"),
    name="pathology_v3",
    exist_ok=True,
    fliplr=0.5,          # safe: disease labels are position-independent
    flipud=0.0,
    mosaic=0.5,
    degrees=5.0,
    translate=0.1,
    scale=0.3,
    hsv_h=0.015,
    hsv_s=0.3,
    hsv_v=0.3,
)
print("best weights:", results.save_dir)

In [ ]:
# ---- validation metrics ----
m = model.val(data=str(WORK / "datasets" / "data_v3.yaml"), imgsz=IMGSZ, split="val", conf=0.001, iou=0.6)
names = [m.names[i] for i in sorted(m.names)]
ap50 = m.box.ap50.tolist()

print(f"mAP50    {m.box.map50:.4f}")
print(f"mAP50-95 {m.box.map:.4f}")
print(f"precision {m.box.mp:.4f} | recall {m.box.mr:.4f}")
print()
for name, ap in sorted(zip(names, ap50), key=lambda t: -t[1]):
    print(f"  {name:22s} AP50 {ap:.3f}")

summary = {
    "model": MODEL, "imgsz": IMGSZ, "epochs": EPOCHS, "batch": BATCH,
    "train_images": len(train_list), "train_unique": len(set(train_list)),
    "val_images": len(val_list),
    "mAP50": float(m.box.map50), "mAP50_95": float(m.box.map),
    "precision": float(m.box.mp), "recall": float(m.box.mr),
    "ap50_per_class": dict(zip(names, map(float, ap50))),
    "train_class_boxes": dict(boxes),
}
(WORK / "pathology_v3_metrics.json").write_text(json.dumps(summary, indent=2))
print("\nwrote pathology_v3_metrics.json")

In [ ]:
# ---- package outputs ----
best = Path(results.save_dir) / "weights" / "best.pt"
shutil.copy2(best, WORK / "pathology_v3_best.pt")
shutil.make_archive(str(WORK / "pathology_v3_run"), "zip", str(results.save_dir))
print("pathology_v3_best.pt:", (WORK / "pathology_v3_best.pt").stat().st_size / 1e6, "MB")
print("artifacts in /kaggle/working: pathology_v3_best.pt, pathology_v3_metrics.json, pathology_v3_run.zip")